<a href="https://colab.research.google.com/github/TOMAKOVALAYAN/github-pages/blob/main/4k_Video_Upscaler_Colab_(Real_ESRGAN).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4k Video Upscaler Colab (Real-ESRGAN)

Adapted from: [Real-ESRGAN](https://github.com/xinntao/Real-ESRGAN)

Made with ❤️ by: [yuvraj108c](https://github.com/yuvraj108c)

Github repository: https://github.com/yuvraj108c/4k-video-upscaler-colab

# 1. Setup (~1 minute)

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU not detected.. Please change runtime to GPU"

from PIL import Image
import cv2, os, subprocess
from tqdm import tqdm

!git clone https://github.com/xinntao/Real-ESRGAN.git
%cd Real-ESRGAN

!pip install -q torch==2.0.1 torchvision==0.15.2 --extra-index-url https://download.pytorch.org/whl/cu118
!pip install -q basicsr facexlib gfpgan ffmpeg ffmpeg-python
!pip install -q -r requirements.txt
!python setup.py develop

!pip install "numpy<2"
mount_drive = False


# 2. Mount drive (optional)

In [ ]:
from google.colab import drive
mount_drive=False #@param{type:"boolean"}

if mount_drive:
  drive.mount('/content/gdrive/')

# 3. Upscale video

- The upscaled video will be saved to `output_dir`
- If google drive is mounted, it will be also saved at `MyDrive/Upscaled Videos (REAL-ESRGAN)`


In [ ]:
import os
import cv2
import subprocess
from google.colab import drive

# --- 0. KÜTÜPHANE TAMİRİ (PATCH) ---
# torchvision güncellemesi nedeniyle bozulan 'functional_tensor' yolunu otomatik düzeltir
degrad_path = "/usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py"
if os.path.exists(degrad_path):
    with open(degrad_path, 'r') as f:
        content = f.read()
    if 'functional_tensor' in content:
        new_content = content.replace('torchvision.transforms.functional_tensor', 'torchvision.transforms.functional')
        with open(degrad_path, 'w') as f:
            f.write(new_content)
        print("✅ Kütüphane uyum hatası giderildi.")

# --- 1. KLASÖR AYARLARI ---
# Ana repo klasörüne geçiş yapıyoruz
%cd /content/Real-ESRGAN

# --- 2. KULLANICI AYARLARI ---
video_path = "/content/drive/MyDrive/vojnik_srece/vojnik.mp4" #@param{type:"string"}
output_dir = "/content/"
resolution = "FHD (1920 x 1080)" # @param ["FHD (1920 x 1080)", "2k (2560 x 1440)", "4k (3840 x 2160)","2 x original", "3 x original", "4 x original"] {type:"string"}
model = "RealESRGAN_x4plus" #@param ["RealESRGAN_x4plus" , "RealESRGAN_x4plus_anime_6B", "realesr-animevideov3"]

# Dosya varlık kontrolü
if not os.path.exists(video_path):
    print(f"❌ HATA: Video dosyası bulunamadı! Lütfen yolu kontrol et: {video_path}")
else:
    # --- 3. VİDEO ANALİZİ ---
    video_capture = cv2.VideoCapture(video_path)
    video_width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    video_height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    video_capture.release()

    match resolution:
        case "FHD (1920 x 1080)": final_width, final_height = 1920, 1080
        case "2k (2560 x 1440)": final_width, final_height = 2560, 1440
        case "4k (3840 x 2160)": final_width, final_height = 3840, 2160
        case "2 x original": final_width, final_height = 2*video_width, 2*video_height
        case "3 x original": final_width, final_height = 3*video_width, 3*video_height
        case "4 x original": final_width, final_height = 4*video_width, 4*video_height

    scale_factor = max(final_width/video_width, final_height/video_height)
    print(f"🚀 Upscale Başlıyor: {video_width}x{video_height} -> {final_width}x{final_height}")

    # --- 4. ANA UPSCALE İŞLEMİ (Inference) ---
    # Bu satır kareleri tek tek netleştirmeye başlar
    !python inference_realesrgan_video.py -n {model} -i '{video_path}' -o '{output_dir}' --outscale {scale_factor}

    # --- 5. SONUÇLARI DÜZENLE VE DRIVE'A AKTAR ---
    video_name_with_ext = os.path.basename(video_path)
    video_name = os.path.splitext(video_name_with_ext)[0]
    upscaled_video_path = f"{output_dir}{video_name}_out.mp4"
    final_video_name = f"{video_name}_HD_FINAL.mp4"
    final_local_path = os.path.join(output_dir, final_video_name)

    if os.path.exists(upscaled_video_path):
        print("🎬 Video boyutları ve formatı ayarlanıyor...")
        ffmpeg_cmd = f"ffmpeg -loglevel error -hwaccel cuda -y -i '{upscaled_video_path}' -c:v h264_nvenc -filter:v 'crop={final_width}:{final_height}:(in_w-{final_width})/2:(in_h-{final_height})/2' -c:v libx264 -pix_fmt yuv420p '{final_local_path}'"
        subprocess.run(ffmpeg_cmd, shell=True)

        drive_folder = "/content/drive/MyDrive/Upscaled_Videos"
        os.makedirs(drive_folder, exist_ok=True)
        final_drive_path = os.path.join(drive_folder, final_video_name)

        print(f"📂 Drive'a taşınıyor...")
        subprocess.run(f"cp '{final_local_path}' '{final_drive_path}'", shell=True)

        print(f"✅ İŞLEM BAŞARIYLA TAMAMLANDI!")
        print(f"📍 Dosyayı burada bulabilirsin: {final_drive_path}")

        # Geçici dosyaları temizle
        if os.path.exists(upscaled_video_path): os.remove(upscaled_video_path)
    else:
        print("❌ HATA: Upscale işlemi sırasında bir sorun oluştu ve çıktı dosyası üretilemedi.")

# 4. Disconnect runtime

In [ ]:
from google.colab import runtime

disconnect_when_finish = False  #@param{type:"boolean"}

if disconnect_when_finish:
  runtime.unassign()